# 02 · 自动求导（Autograd）

> PyTorch 最核心的能力：**自动计算梯度**。
> 你只需要写前向传播，反向传播由 autograd 自动完成。

## 2.1 为什么需要自动求导

神经网络的训练 = 用梯度下降不断调整参数：

$$
\theta \leftarrow \theta - \eta \cdot \frac{\partial L}{\partial \theta}
$$

手工推导几百万参数的梯度是不现实的。autograd 通过记录**计算图**，用反向传播自动求出所有参数的梯度。

## 2.2 requires_grad：告诉 PyTorch "这个要算梯度"

- `requires_grad=True` 的张量，参与运算后会产生计算图。
- 只有**浮点类型**张量才能要求梯度。

In [ ]:
import torch

x = torch.tensor(2.0, requires_grad=True)
print("requires_grad:", x.requires_grad)

# 普通张量没有梯度
w = torch.tensor(3.0)
print("普通张量 requires_grad:", w.requires_grad)

# 整数不能要求梯度
try:
    t = torch.tensor(3, requires_grad=True)
except Exception as e:
    print("整数报错:", e)

## 2.3 最简单的例子：y = x²

设 $y = x^2$，则 $\frac{dy}{dx} = 2x$。当 $x=2$ 时梯度应为 4。

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2

print("y =", y)
print("y 是否带梯度:", y.requires_grad)

# 反向传播：从 y 出发，自动求出所有 requires_grad 张量的梯度
y.backward()

# x.grad 保存了 dy/dx
print("x.grad =", x.grad, " (期望 2*x = 4)")

## 2.4 计算图

PyTorch 采用**动态计算图**（define-by-run）：每次前向传播都会重新构建图。

```
x (requires_grad)
   |
   v
  x**2  ---> y (grad_fn=PowBackward0)
```

每个带梯度的张量都有 `grad_fn`，记录它是怎么算出来的。

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
a = x * 3
b = a + 1
y = b ** 2

print("x.grad_fn :", x.grad_fn)     # 叶子节点，None
print("a.grad_fn :", a.grad_fn)
print("b.grad_fn :", b.grad_fn)
print("y.grad_fn :", y.grad_fn)
print("y.is_leaf :", y.is_leaf, " x.is_leaf :", x.is_leaf)
print("\n验证: y=((3x+1)^2). 当 x=2, y =", y.item())

### 叶子节点（leaf）

- **叶子节点**：用户直接创建的张量（如 `x`），它们的梯度保存在 `.grad`。
- **非叶子节点**：运算得到的中间结果，默认不保存梯度（省内存）。
- 想要查看中间张量的梯度，用 `.retain_grad()`。

中间结果占内存，所以默认不保留是合理的。

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
a = x * 3
a.retain_grad()          # 手动要求保留 a 的梯度
y = a ** 2
y.backward()

print("x.grad =", x.grad)   # dy/dx = 2*(3x) * 3 = 2*6*3 = 36
print("a.grad =", a.grad)   # dy/da = 2a = 12

## 2.5 标量 vs 非标量 backward

### 标量（0 维）可以直接 backward

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = (x ** 2).sum()          # 通过 sum 变成标量
y.backward()
print("x.grad =", x.grad, " (期望 2x = [2,4,6])")

### 非标量必须提供 gradient 参数

当输出不是标量时，`backward(gradient)` 中的 `gradient` 相当于
"y 各元素对最终损失的权重"，等价于计算 `(gradient * y).sum().backward()`。

最典型场景：`outputs` 是 (N, C) 的 logits，`gradient` 是损失对它的梯度，
但通常我们用 `loss.backward()`（loss 已是标量），所以少见。

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x ** 2                              # 非标量
print("y =", y)

# 直接 backward 会报错
try:
    y.backward()
except Exception as e:
    print("报错:", e)

# 提供梯度（相当于权重）
y.backward(gradient=torch.tensor([1.0, 1.0, 1.0]))
print("带 gradient 的 x.grad =", x.grad)   # 与 sum 结果一致

# 等价写法：换个新的计算图（y 的图已在上一步释放）
x.grad.zero_()
y2 = x ** 2
y2.sum().backward()
print("sum().backward() 的 x.grad =", x.grad)

## 2.6 梯度累积与清零

**关键规则**：`backward()` 会把梯度**累加**到 `.grad`，而不是覆盖。

这是容易踩的坑：普通训练循环里每步必须 `zero_grad()`。

In [ ]:
x = torch.tensor(2.0, requires_grad=True)

for i in range(3):
    y = x ** 2
    y.backward()
    print(f"第 {i+1} 次 backward 后 x.grad = {x.grad}")

print("\n注意：梯度被累加了（2x=4，累积 3 次 = 12）")

# 清零
x.grad.zero_()
print("zero_() 后 x.grad =", x.grad)

### 梯度累加是有用的技巧

当显存不足、无法用大 batch 时，可以：
- 用几个小 batch 分别 forward/backward（**不清零**），
- 累加几个 batch 后再 `optimizer.step()` 一次，等效于大 batch。

后面 11 节会演示。

## 2.7 停止追踪梯度

### with torch.no_grad()

评估/推理时不需要梯度，用 `no_grad()` 可以：
1. 省显存（不保存中间结果）
2. 加速

```python
model.eval()
with torch.no_grad():
    pred = model(x)
```

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

# 默认：会被追踪
y = x ** 2
print("普通运算 y.requires_grad:", y.requires_grad)

# no_grad：不追踪
with torch.no_grad():
    z = x ** 2
print("no_grad 内 z.requires_grad:", z.requires_grad)
print("但 z 仍然是正常张量:", z)

# 也可以装饰器写法
@torch.no_grad()
def compute(x):
    return x * 2

print("装饰器:", compute(x).requires_grad)

### detach()：从计算图分离

`detach()` 返回一个**共享数据但脱离了计算图**的新张量。
常用于：
- 把张量转成 numpy 画图
- 目标网络 / 停止某些分支的梯度（如 GAN、强化学习）

In [ ]:
x = torch.tensor([1.0, 2.0], requires_grad=True)
y = x ** 2

d = y.detach()
print("y.requires_grad:", y.requires_grad)
print("d.requires_grad:", d.requires_grad)

# 修改 detach 的张量会影响原张量（共享内存）
d[0] = 99
print("y 也变了:", y)

# 常见用途：损失数值记录
x = torch.tensor(3.0, requires_grad=True)
loss = x ** 2
print("\nloss.item() 安全取值:", loss.item())

### torch.enable_grad()

在 `no_grad()` 内部临时恢复追踪：

```python
with torch.no_grad():
    ...
    with torch.enable_grad():
        ...   # 这里会追踪
```

## 2.8 反向传播的完整例子：手算梯度下降

用 autograd 实现梯度下降，不用 `nn` 和 `optim`，彻底理解原理。

目标：拟合 $y = 2x + 1$。

In [ ]:
import torch

torch.manual_seed(0)

# 数据
X = torch.rand(50, 1) * 4 - 2
y = 2.0 * X + 1.0 + 0.05 * torch.randn(50, 1)

# 手写参数（叶子张量）
w = torch.randn(1, 1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

lr = 0.1
for epoch in range(50):
    # 前向
    y_pred = X @ w + b
    loss = ((y_pred - y) ** 2).mean()

    # 反向
    loss.backward()

    # 手动更新（在 no_grad 下操作，避免更新被记录进计算图）
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad

    # 手动清零
    w.grad.zero_()
    b.grad.zero_()

    if (epoch + 1) % 10 == 0:
        print(f"epoch {epoch+1:2d} | loss {loss.item():.5f} | w {w.item():.3f} | b {b.item():.3f}")

print("\n真实 w=2, b=1")

> **注意**：更新参数时必须 `with torch.no_grad()`，
> 否则 `w -= ...` 这个操作会进入计算图，导致错误。

`optimizer.step()` 内部已经帮你做了这件事，所以平时不用手写。

## 2.9 关于训练中的常见报错

| 报错 | 原因 | 解决 |
|---|---|---|
| `element 0 of tensors does not require grad` | 参数没设 `requires_grad` | 用 `nn.Parameter` 或检查模型参数 |
| `Trying to backward through the graph a second time` | 同一个图 backward 两次或没清零 | 重新 forward，或 `retain_graph=True` |
| `a leaf Variable that requires grad is being used in an in-place operation` | 在叶子张量上原地修改 | 用 `with torch.no_grad()` |

`retain_graph=True` 示例（一般用不到，除非需要多次 backward）：

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2
y.backward(retain_graph=True)      # 保留计算图
y.backward()
print("x.grad =", x.grad)          # 4 + 4 = 8（累加）

## 2.10 练习

**练习 1**：对 $f(x) = x^3 + 2x$ 求 $x=1.5$ 处的导数，用 autograd 验证
（手算：$3x^2+2 = 8.75$）。

**练习 2**：对 $z = x \cdot y + y^2$，$x=3, y=2$，
分别求 $\partial z/\partial x$ 和 $\partial z/\partial y$。

**练习 3**：用 autograd 对 $f(x) = (x-3)^2$ 做 20 步梯度下降（lr=0.1），
看 x 是否收敛到 3。

In [ ]:
# 练习 1
x = torch.tensor(1.5, requires_grad=True)
f = x ** 3 + 2 * x
f.backward()
print("练习1 导数:", x.grad.item(), "(期望 8.75)")

# 练习 2
x = torch.tensor(3.0, requires_grad=True)
y = torch.tensor(2.0, requires_grad=True)
z = x * y + y ** 2
z.backward()
print("练习2 dz/dx:", x.grad.item(), "(期望 y=2)")
print("练习2 dz/dy:", y.grad.item(), "(期望 x+2y=7)")

# 练习 3
x = torch.tensor(0.0, requires_grad=True)
for _ in range(20):
    f = (x - 3) ** 2
    f.backward()
    with torch.no_grad():
        x -= 0.1 * x.grad
    x.grad.zero_()
print("练习3 收敛到 x =", x.item())

## 2.11 小结

- `requires_grad=True` 开启追踪；只有浮点张量可用。
- `loss.backward()` 自动求所有叶子节点的梯度，存到 `.grad`。
- 梯度会**累加**，训练每步要 `zero_grad()`。
- 推理用 `with torch.no_grad()` 省内存加速。
- 想脱离计算图用 `.detach()`。
- 动态图：每次前向都重新构建。

下一节：`03_构建神经网络nn.Module.ipynb`